In [6]:
print("Spotify Churn Model is ready!")

Spotify Churn Model is ready!


In [7]:
from pathlib import Path
import pandas as pd
import sqlite3

data_dir = Path("data")

csv_files = list(data_dir.glob("*.csv"))

print("CSV files found:", len(csv_files))

for file in sorted(csv_files):
    print(file.name)

CSV files found: 5
ad_events.csv
feature_table.csv
listening_events.csv
subscription_events.csv
users.csv


In [8]:
conn = sqlite3.connect(":memory:")

tables = [
    "users",
    "listening_events",
    "subscription_events",
    "ad_events",
    "feature_table"
]

for table in tables:
    df = pd.read_csv(data_dir / f"{table}.csv")
    df.to_sql(table, conn, if_exists="replace", index=False)

    print(f"{table}: {df.shape[0]:,} rows, {df.shape[1]} columns")

users: 50,000 rows, 11 columns
listening_events: 1,072,063 rows, 15 columns
subscription_events: 104,498 rows, 10 columns
ad_events: 310,438 rows, 10 columns
feature_table: 50,000 rows, 53 columns


In [9]:
pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
""", conn)

,name
0,users
1,listening_events
2,subscription_events
3,ad_events
4,feature_table


In [11]:
# Execute SQL queries and return results as a DataFrame
def sql(query):
    return pd.read_sql_query(query, conn)

# Execute SQL scripts (e.g., CREATE VIEW, DROP VIEW)
def run_script(query):
    conn.executescript(query)
    print("SQL script executed successfully.")

## Section 0 | Data Granularity Check

Check the number of records and unique users in each table.

In [12]:
# Check row counts and distinct users across all tables
query = """
SELECT 'users' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM users

UNION ALL

SELECT 'listening_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM listening_events

UNION ALL

SELECT 'subscription_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM subscription_events

UNION ALL

SELECT 'ad_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM ad_events

UNION ALL

SELECT 'feature_table' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM feature_table;
"""

sql(query)

,table_name,row_count,user_count
0,users,50000,50000
1,listening_events,1072063,48187
2,subscription_events,104498,34640
3,ad_events,310438,31000
4,feature_table,50000,50000


## Section 1 | Retention Overview

Establish baseline metrics for user engagement, churn, and conversion.

In [13]:
# Establish baseline user engagement and conversion metrics
query = """
SELECT
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN active_days_30d > 0 THEN user_id END) AS active_users_30d,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_minutes_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(cancel_30d), 4) AS cancel_rate_30d
FROM feature_table
;"""

sql(query)

,total_users,active_users_30d,avg_active_days_30d,avg_listen_minutes_30d,churn_rate_14d,paid_conversion_rate_30d,cancel_rate_30d
0,50000,44118,6.43,38.53,0.4756,0.1739,0.0129


**Insight:** 47.56% of users had no listening activity in the following 14 days, while the 30-day cancellation rate was only 1.29%. Listening inactivity and subscription cancellation represent different outcomes.

**Next Action:** Segment users by subscription type to identify which groups contribute most to churn.

## Section 2 | Subscription Segmentation

Compare engagement, churn, and conversion across subscription types.

In [15]:
# Compare user behavior and churn across subscription types
query = """SELECT
    subscription_type,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_minutes_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(cancel_30d), 4) AS cancel_rate_30d
FROM feature_table
GROUP BY subscription_type
ORDER BY users DESC
;"""

sql(query)

,subscription_type,users,avg_active_days_30d,avg_listen_minutes_30d,churn_rate_14d,paid_conversion_rate_30d,cancel_rate_30d
0,free,30006,5.18,29.72,0.5735,0.0010,0.0180
1,individual_premium,14555,8.43,52.46,0.3198,0.4596,0.0000
2,family_premium,3247,8.59,53.34,0.3163,0.4666,0.0000
3,trial,1249,6.30,37.57,0.4652,0.0000,0.0857
4,student_premium,943,8.54,54.04,0.3277,0.4889,0.0000


**Insight:** Free users have the highest 14-day churn rate (57.35%)
and account for approximately 72.4% of all churned users.
Premium users show higher engagement and lower churn rates.

**Action:** Prioritize free-user retention analysis and investigate
trial-user cancellation patterns.

## Section 3 | Low Activity & Churn Risk

Hypothesis: Users with fewer active days are more likely to churn.

In [16]:
# Test the relationship between activity level and churn
query = """WITH user_activity AS (
    SELECT
        user_id,
        active_days_30d,
        listen_minutes_30d,
        paid_conversion_30d,
        churn_label_14d,
        CASE
            WHEN active_days_30d = 0 THEN 'inactive'
            WHEN active_days_30d <= 3 THEN 'low'
            WHEN active_days_30d <= 10 THEN 'medium'
            ELSE 'high'
        END AS active_level
    FROM feature_table
)
SELECT
    active_level,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_minutes_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d
FROM user_activity
GROUP BY active_level
ORDER BY
    CASE active_level
        WHEN 'inactive' THEN 1
        WHEN 'low' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'high' THEN 4
    END
;"""

sql(query)

,active_level,users,avg_active_days_30d,avg_listen_minutes_30d,churn_rate_14d,paid_conversion_rate_30d
0,inactive,5882,0.00,0.00,0.7724,0.1000
1,low,15070,1.90,8.56,0.6663,0.1291
2,medium,18135,6.47,35.28,0.4374,0.1822
3,high,10913,16.10,106.09,0.1156,0.2619


**Insight:** Users with lower activity levels show substantially higher
churn rates. Inactive users have a 77.24% churn rate, compared with
11.56% among highly active users.

**Action:** Prioritize inactive and low-activity users for retention
analysis and test personalized re-engagement strategies.

## Section 4 | Ad Load & Retention

Hypothesis: Higher ad exposure is associated with increased churn risk.

In [17]:
# Compare ad exposure and churn among free and trial users
query = """WITH ad_base AS (
    SELECT
        user_id,
        subscription_type,
        active_level_30d,
        active_days_30d,
        ad_impressions_30d,
        ad_impressions_per_active_day,
        churn_label_14d,
        cancel_30d,
        CASE
            WHEN ad_impressions_per_active_day = 0 THEN 'no_ads'
            WHEN ad_impressions_per_active_day <= 2 THEN 'low'
            WHEN ad_impressions_per_active_day <= 5 THEN 'medium'
            ELSE 'high'
        END AS ad_load_group
    FROM feature_table
    WHERE subscription_type IN ('free', 'trial')
)
SELECT
    ad_load_group,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(ad_impressions_per_active_day), 2) AS avg_ads_per_active_day,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(cancel_30d), 4) AS cancel_rate_30d
FROM ad_base
GROUP BY ad_load_group
ORDER BY
    CASE ad_load_group
        WHEN 'no_ads' THEN 1
        WHEN 'low' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'high' THEN 4
    END
;"""

sql(query)

,ad_load_group,users,avg_active_days_30d,avg_ads_per_active_day,churn_rate_14d,cancel_rate_30d
0,no_ads,7903,0.83,0.00,0.7681,0.0170
1,low,22063,6.92,0.87,0.4928,0.0224
2,medium,1226,3.26,3.09,0.6525,0.0131
3,high,63,1.02,6.56,0.7302,0.0159


**Insight:** Users with no ad exposure have the highest churn rate
(76.81%), but also the lowest engagement (0.83 active days).
This suggests that activity level may confound the relationship
between ad exposure and churn.

**Next Action:** Compare ad load groups within the same activity level
to better isolate the association between ad exposure and churn.

## Section 5 | Ad Load Analysis by Activity Level

Compare churn rates across ad load groups within the same activity segment.

In [18]:
# Compare ad load and churn within the same activity level
query = """WITH ad_base AS (
    SELECT
        user_id,
        active_level_30d,
        ad_impressions_per_active_day,
        churn_label_14d,
        CASE
            WHEN ad_impressions_per_active_day = 0 THEN 'no_ads'
            WHEN ad_impressions_per_active_day <= 2 THEN 'low'
            WHEN ad_impressions_per_active_day <= 5 THEN 'medium'
            ELSE 'high'
        END AS ad_load_group
    FROM feature_table
    WHERE subscription_type IN ('free', 'trial')
)
SELECT
    active_level_30d,
    ad_load_group,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(ad_impressions_per_active_day), 2) AS avg_ads_per_active_day,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM ad_base
GROUP BY active_level_30d, ad_load_group
HAVING COUNT(DISTINCT user_id) >= 50
ORDER BY active_level_30d, ad_load_group
;"""

sql(query)

,active_level_30d,ad_load_group,users,avg_ads_per_active_day,churn_rate_14d
0,high,low,4656,0.82,0.1622
1,high,medium,92,2.35,0.1413
2,inactive,no_ads,4625,0.00,0.8067
3,low,high,63,6.56,0.7302
4,low,low,6993,1.06,0.6998
5,low,medium,962,3.27,0.7152
6,low,no_ads,2870,0.00,0.7307
7,medium,low,10414,0.77,0.5016
8,medium,medium,172,2.50,0.5756
9,medium,no_ads,399,0.00,0.6040


**Insight:** After segmenting users by activity level, the relationship
between ad exposure and churn is not consistently monotonic.
Activity level remains a strong indicator of churn risk, while
differences across ad load groups vary by segment.

**Next Action:** Further investigate ad exposure while accounting for
user engagement and other potential confounding factors.

## Section 6 | Trial Exposure & Paid Conversion

Hypothesis: Trial exposure is associated with higher paid conversion rates.

In [19]:
# Compare conversion and churn between trial-exposed and unexposed users
query = """SELECT
    trial_exposed_30d,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM feature_table
WHERE subscription_type IN ('free', 'trial')
GROUP BY trial_exposed_30d
ORDER BY trial_exposed_30d
;"""

sql(query)

,trial_exposed_30d,users,avg_active_days_30d,paid_conversion_rate_30d,churn_rate_14d
0,0,24052,5.07,0.0006,0.5795
1,1,7203,5.72,0.0021,0.5345


**Insight:** Trial-exposed users show a higher paid conversion rate
(0.21% vs. 0.06%) and a lower churn rate (53.45% vs. 57.95%).
However, exposed users are also slightly more active, so these
differences do not establish a causal effect.

**Next Action:** Examine the subscription conversion funnel to
identify where users drop off before becoming paid subscribers.

## Section 7 | Subscription Conversion Funnel

Explore user engagement, trial exposure, trial starts, and paid conversion.

In [20]:
# Count users at each stage of the subscription journey
query = """SELECT
    '1_free_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial')

UNION ALL

SELECT
    '2_engaged_free_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial')
  AND active_days_30d >= 4

UNION ALL

SELECT
    '3_trial_exposed_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial')
  AND trial_exposed_30d = 1

UNION ALL

SELECT
    '4_trial_started_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial')
  AND trial_started_30d = 1

UNION ALL

SELECT
    '5_paid_converted_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial')
  AND paid_conversion_30d = 1
;"""

sql(query)

,funnel_step,users
0,1_free_users,31255
1,2_engaged_free_users,15742
2,3_trial_exposed_users,7203
3,4_trial_started_users,2681
4,5_paid_converted_users,30


**Insight:** Among 31,255 free and trial users, 7,203 had trial
exposure, 2,681 started a trial, and only 30 were flagged as
paid conversions within the dataset's 30-day metric.

**Next Action:** Investigate the low paid conversion count and
validate the funnel definitions before calculating step-to-step
conversion rates.

## Section 8 | Device & Paid Conversion

Hypothesis: Mobile users have higher paid conversion rates than desktop users.

In [21]:
# Compare engagement, trial exposure, and conversion by device
query = """SELECT
    primary_device,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(trial_exposed_30d), 4) AS trial_exposure_rate_30d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM feature_table
GROUP BY primary_device
ORDER BY paid_conversion_rate_30d DESC
;"""

sql(query)

,primary_device,users,avg_active_days_30d,trial_exposure_rate_30d,paid_conversion_rate_30d,churn_rate_14d
0,mobile_ios,19806,6.46,0.2689,0.1911,0.4700
1,mobile_android,17510,6.41,0.2558,0.1716,0.4787
2,smart_speaker,1545,6.78,0.2479,0.1702,0.4563
3,desktop,6548,6.41,0.2434,0.1513,0.4821
4,web,4591,6.34,0.2398,0.1425,0.4851


**Insight:** Mobile users show higher paid conversion metrics than
desktop and web users. iOS has the highest rate (19.11%), compared
with 14.25% for web users.

**Next Action:** Investigate device-specific user journeys and
conversion patterns, while accounting for subscription status
and engagement differences.

## Section 9 | Content Interaction & Retention

Hypothesis: Users who interact with content are less likely to churn.

In [22]:
# Compare churn between users with and without content interactions
query = """WITH content_base AS (
    SELECT
        user_id,
        playlist_adds_30d,
        liked_songs_30d,
        search_events_30d,
        churn_label_14d,
        CASE
            WHEN playlist_adds_30d > 0 OR liked_songs_30d > 0 OR search_events_30d > 0
                THEN 'has_content_interaction'
            ELSE 'no_content_interaction'
        END AS content_interaction_group
    FROM feature_table
)
SELECT
    content_interaction_group,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(playlist_adds_30d), 2) AS avg_playlist_adds_30d,
    ROUND(AVG(liked_songs_30d), 2) AS avg_liked_songs_30d,
    ROUND(AVG(search_events_30d), 2) AS avg_search_events_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM content_base
GROUP BY content_interaction_group
;"""

sql(query)

,content_interaction_group,users,avg_playlist_adds_30d,avg_liked_songs_30d,avg_search_events_30d,churn_rate_14d
0,has_content_interaction,33720,0.72,1.03,2.61,0.3696
1,no_content_interaction,16280,0.00,0.00,0.00,0.6950


**Insight:** Users with content interactions have a substantially
lower 14-day churn rate (36.96%) than users without interactions
(69.50%), a difference of 32.54 percentage points.

**Next Action:** Include content engagement features such as
playlist additions, liked songs, and search activity in the
churn prediction model.